# FinMark Predictive Model
### Will a customer make a purchase? Let's build a model to predict that!
---
This notebook loads 3 datasets, cleans them up, joins them together, and uses a machine learning algorithm called **Logistic Regression** to predict whether a customer is likely to make a purchase or not.

## Step 1 — Load the tools we need
Before we do anything, we need to import the libraries (think of these as toolboxes) that help us work with data and build the model.

In [ ]:
# pandas helps us work with tables of data (like Excel but in Python)
import pandas as pd

# numpy helps us do math stuff like finding averages
import numpy as np

# these are the machine learning tools we need
from sklearn.linear_model import LogisticRegression   # the model we will use
from sklearn.model_selection import train_test_split  # splits data into training and testing
from sklearn.preprocessing import StandardScaler, LabelEncoder  # helps prepare the data
from sklearn.metrics import (
    accuracy_score,        # tells us how accurate the model is
    classification_report, # gives us a detailed performance report
    confusion_matrix,      # shows us where the model got confused
    roc_auc_score          # another way to measure how good the model is
)

# this just hides some annoying warning messages
import warnings
warnings.filterwarnings('ignore')

print('All tools loaded successfully!')

---
## Step 2 — Tell Python where our files are and load them
We point Python to the folder where all our CSV files are saved, then we load all 3 datasets.

In [ ]:
import os

# this tells Python which folder to look in for our files
# and where to save any new files we create
os.chdir(r'C:\Users\Matti\Documents\School')

# now we load each CSV file into a variable
# think of each variable as a table we can look at and work with
customers_df    = pd.read_csv('customers_data.csv')
products_df     = pd.read_csv('products_data.csv')
transactions_df = pd.read_csv('transactions_data.csv')

# let's print a quick summary to confirm everything loaded
print('Datasets loaded!')
print(f'   customers_data:    {customers_df.shape[0]} rows, {customers_df.shape[1]} columns')
print(f'   products_data:     {products_df.shape[0]} rows, {products_df.shape[1]} columns')
print(f'   transactions_data: {transactions_df.shape[0]} rows, {transactions_df.shape[1]} columns')

---
## Step 3 — Take a look at the data
Before we do anything, it's important to understand what columns we have and what type of data is in them (numbers, text, dates, etc.).

In [ ]:
# let's check the column names, data types, and how many values are missing
for name, df in [('customers_data', customers_df), ('products_data', products_df), ('transactions_data', transactions_df)]:
    print(f'\n--- {name} ---')
    print(f'Columns and their data types:')
    print(df.dtypes)
    print(f'\nHow many values are missing per column:')
    print(df.isnull().sum())

In [ ]:
# let's also preview the first few rows of each dataset
# so we can see what the actual data looks like
print('--- First 5 rows of customers_data ---')
display(customers_df.head())

print('--- First 5 rows of products_data ---')
display(products_df.head())

print('--- First 5 rows of transactions_data ---')
display(transactions_df.head())

---
## Step 4 — Clean the data
Real data is messy. Some rows have missing values (shown as NaN which means "Not a Number"). We need to fix these before we can use the data.

Our plan:
- If a row is missing its ID number, we drop that row entirely (we can't use it without knowing who it belongs to)
- If a number column has missing values, we fill them with the median (the middle value)
- If a text column has missing values, we fill them with the word 'Unknown'

In [ ]:
# --- Clean customers_data ---

# make a copy so we don't accidentally change the original
customers_clean = customers_df.copy()

# drop any row that doesn't have a Company_ID
# (we can't use a customer record if we don't know who it is)
customers_clean = customers_clean.dropna(subset=['Company_ID'])

# fill missing profit values with the median profit
customers_clean['Company_Profit'] = customers_clean['Company_Profit'].fillna(customers_clean['Company_Profit'].median())

# convert Company_ID from decimal (1.0) to whole number (1)
customers_clean['Company_ID'] = customers_clean['Company_ID'].astype(int)

print(f'customers_data cleaned: missing values went from {customers_df.isnull().sum().sum()} to {customers_clean.isnull().sum().sum()}')

In [ ]:
# --- Clean products_data ---

products_clean = products_df.copy()

# drop rows with no Product_ID
products_clean = products_clean.dropna(subset=['Product_ID'])

# the Product_Price column has a '?' symbol in front of the numbers
# we need to remove that symbol so Python can treat it as a number
products_clean['Product_Price'] = (
    products_clean['Product_Price']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)  # remove anything that isn't a number or dot
    .replace('', np.nan)                      # if the result is empty, mark it as missing
)
products_clean['Product_Price'] = pd.to_numeric(products_clean['Product_Price'], errors='coerce')
products_clean['Product_Price'] = products_clean['Product_Price'].fillna(products_clean['Product_Price'].median())

# convert Product_ID to a whole number
products_clean['Product_ID'] = products_clean['Product_ID'].astype(int)

print(f'products_data cleaned: missing values went from {products_df.isnull().sum().sum()} to {products_clean.isnull().sum().sum()}')

In [ ]:
# --- Clean transactions_data ---

transactions_clean = transactions_df.copy()

# remove the unnamed column that pandas added automatically (it's just row numbers, we don't need it)
transactions_clean = transactions_clean.drop(columns=['Unnamed: 0'], errors='ignore')

# drop rows that are missing their ID columns
# (a transaction without an ID is useless)
transactions_clean = transactions_clean.dropna(subset=['Transaction_ID', 'Company_ID', 'Product_ID'])

# fill missing number columns with their median values
for col in ['Quantity', 'Product_Price', 'Total_Cost']:
    transactions_clean[col] = transactions_clean[col].fillna(transactions_clean[col].median())

# convert ID columns to whole numbers
for col in ['Transaction_ID', 'Company_ID', 'Product_ID']:
    transactions_clean[col] = transactions_clean[col].astype(int)

print(f'transactions_data cleaned: missing values went from {transactions_df.isnull().sum().sum()} to {transactions_clean.isnull().sum().sum()}')

---
## Step 5 — Join the 3 datasets together
Right now our data is spread across 3 separate tables. We need to combine them into one big table so the model can see all the information at once.

We join them using the ID columns they share — like connecting puzzle pieces.

In [ ]:
# first join transactions with customers using Company_ID
df = transactions_clean.merge(customers_clean, on='Company_ID', how='left')

# then join with products using Product_ID
df = df.merge(products_clean[['Product_ID', 'Product_Name']], on='Product_ID', how='left')

print(f'Combined dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
display(df.head())

---
## Step 6 — Fix any leftover missing values from the join
When we joined the tables, some rows couldn't find a match — those came back as NaN again.
We clean those up here.

In [ ]:
# fill any remaining missing numbers with the column median
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

# fill any remaining missing text with 'Unknown'
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('Unknown')

print(f'Leftover missing values after join: {df.isnull().sum().sum()}')
print('All clean and ready to go!')

---
## Step 7 — Create the answer column (target variable)
Our model needs to know what it's trying to predict. We create a new column called **purchased**.
- If Quantity is greater than 0, the customer made a purchase → we write **1**
- If Quantity is 0, no purchase was made → we write **0**

In [ ]:
# create the purchased column: 1 = bought something, 0 = didn't buy
df['purchased'] = (df['Quantity'] > 0).astype(int)

# let's see how many purchases vs non-purchases we have
print('How many purchased (1) vs not purchased (0):')
print(df['purchased'].value_counts())
print(f'\nOverall purchase rate: {df["purchased"].mean()*100:.1f}%')

---
## Step 8 — Choose which columns to use as inputs for the model
We can't just throw every column at the model. We pick the ones that are most likely to help it figure out if someone will make a purchase.

In [ ]:
# these are the columns (features) we will use to train the model
feature_cols = [
    'Company_Profit',  # how profitable the company is
    'Product_Price',   # how expensive the product is
    'Quantity',        # how many items were in the transaction
    'Total_Cost',      # the total amount spent
    'Product_Name',    # which product it was
]

# only keep the columns that actually exist in our dataset
feature_cols = [col for col in feature_cols if col in df.columns]

print(f'We are using these {len(feature_cols)} features to train the model:')
for col in feature_cols:
    print(f'   - {col}')

# X = the input data (what the model learns from)
# y = the answer column (what the model is trying to predict)
X = df[feature_cols].copy()
y = df['purchased']

---
## Step 9 — Convert text columns into numbers
Machine learning models can only work with numbers, not text. So we convert text columns like Product_Name into numbers using a LabelEncoder (it just assigns a number to each unique text value).

In [ ]:
# create the encoder tool
le = LabelEncoder()

# find all columns that contain text
text_columns = X.select_dtypes(include='object').columns.tolist()

# convert each text column to numbers
for col in text_columns:
    X[col] = le.fit_transform(X[col].astype(str))

print(f'Converted text columns to numbers: {text_columns}')

# double check there are no missing values left before training
print(f'\nFinal check — missing values in our input data: {X.isnull().sum().sum()}')
print('We are ready to train!' if X.isnull().sum().sum() == 0 else 'Warning: still have missing values!')

---
## Step 10 — Split the data into training and testing sets
We split the data into two parts:
- **Training set (80%)** — the model learns from this
- **Test set (20%)** — we use this to check how well the model performs on data it has never seen before

In [ ]:
# split the data — 80% for training, 20% for testing
# random_state=42 just means the split will be the same every time we run this
# stratify=y makes sure both splits have a similar ratio of 1s and 0s
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:  {X_train.shape[0]} rows (model learns from these)')
print(f'Test set:      {X_test.shape[0]} rows (we test the model on these)')

---
## Step 11 — Scale the numbers so they are all on a similar range
Some columns have very large numbers (like Total_Cost in the millions) and some have small numbers (like Quantity). Scaling puts them all on the same scale so the model doesn't get confused by the size differences.

In [ ]:
# create the scaler tool
scaler = StandardScaler()

# fit the scaler on training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# only transform the test data (don't fit on it — the model shouldn't peek at test data)
X_test_scaled = scaler.transform(X_test)

print('All numbers are now scaled and ready!')

---
## Step 12 — Train the model!
This is the exciting part. We create a Logistic Regression model and train it on our data. Training means the model looks at all the examples and figures out patterns — like "when Total_Cost is high and Quantity is above 5, it's usually a purchase".

In [ ]:
# create the Logistic Regression model
model = LogisticRegression(random_state=42, max_iter=1000)

# train it using the training data
# X_train_scaled = the input columns, y_train = the correct answers
model.fit(X_train_scaled, y_train)

print('Model trained successfully!')

---
## Step 13 — Make predictions and see how well the model did
Now we ask the model to predict purchases on the test set (data it has never seen before) and compare its predictions to the real answers.

In [ ]:
# ask the model to make predictions on the test data
y_pred = model.predict(X_test_scaled)

# also get the probability (how confident the model is) for each prediction
y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]

# calculate how accurate the model is
accuracy = accuracy_score(y_test, y_pred)
roc_auc  = roc_auc_score(y_test, y_pred_prob)

# print the results
print('=' * 45)
print('         MODEL EVALUATION RESULTS')
print('=' * 45)
print(f'  Accuracy:      {accuracy*100:.2f}%')
print(f'  ROC-AUC Score: {roc_auc:.4f}  (closer to 1.0 is better)')
print('=' * 45)
print('\nDetailed Report (precision, recall, f1-score):')
print(classification_report(y_test, y_pred, target_names=['Not Purchased', 'Purchased']))
print('Confusion Matrix (rows = actual, columns = predicted):')
print(confusion_matrix(y_test, y_pred))

---
## Step 14 — Which features mattered most?
Logistic Regression gives each feature a "coefficient" (a score). A higher positive score means that feature strongly pushed the prediction toward "purchased". A negative score means it pushed toward "not purchased".

In [ ]:
# create a table showing each feature and its importance score
importance_table = pd.DataFrame({
    'Feature': feature_cols,
    'Score': model.coef_[0]
}).sort_values('Score', ascending=False)

print('Feature importance scores (higher = more influence on predicting a purchase):')
print(importance_table.to_string(index=False))

---
## Step 15 — Save everything
Let's save the predictions and the cleaned datasets so we can use them later or upload to GitHub.

In [ ]:
# save the model's predictions to a CSV file
results_df = X_test.copy()
results_df['actual']               = y_test.values       # what actually happened
results_df['predicted']            = y_pred               # what the model predicted
results_df['purchase_probability'] = y_pred_prob          # how confident the model was
results_df.to_csv('finmark_predictions.csv', index=False)

# save the cleaned versions of the original datasets
customers_clean.to_csv('customers_data_clean.csv', index=False)
products_clean.to_csv('products_data_clean.csv', index=False)
transactions_clean.to_csv('transactions_data_clean.csv', index=False)

print('All files saved to C:\\Users\\Matti\\Documents\\School')
print('')
print('Files saved:')
print('   finmark_predictions.csv     <- model results')
print('   customers_data_clean.csv    <- cleaned customers')
print('   products_data_clean.csv     <- cleaned products')
print('   transactions_data_clean.csv <- cleaned transactions')